# Phase 6: Walk-Forward Validation

Proper time-series backtesting with the proven 26-feature baseline

## Setup

In [ ]:
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Connect to DuckDB
con = duckdb.connect('../../../data/investing/investing.duckdb', read_only=True)

print('Setting up walk-forward cross-validation')
print('Strategy: Train 5 years, test 1 year, roll forward')
print('Using 26-feature baseline (proven ~65% accuracy)')

## Load and Engineer All Historical Features

**26-feature baseline: SPY momentum/volatility, asset class signals, breadth, fear gauge**

In [ ]:
# Load all data
spy_df = con.execute("""
    SELECT date, close FROM silver_macro_prices
    WHERE symbol = 'SPY' ORDER BY date
""").df()
spy_df['date'] = pd.to_datetime(spy_df['date'])
spy = spy_df.set_index('date')['close'].sort_index()

# Initialize feature matrix
feat = pd.DataFrame(index=spy.index)

# 1A. SPY features
feat['ret_spy_1m'] = spy.pct_change(21)
feat['ret_spy_3m'] = spy.pct_change(63)
feat['ret_spy_6m'] = spy.pct_change(126)
feat['ret_spy_12m'] = spy.pct_change(252)
feat['rvol_spy_63d'] = spy.pct_change().rolling(63).std()

vol_recent = spy.pct_change().rolling(21).std()
vol_trailing = spy.pct_change().rolling(126).std()
feat['vol_ratio_spy'] = vol_recent / vol_trailing

# 1B. Asset class signals
symbols_map = {'QQQ': 'qqq', 'XLE': 'xle', 'GLD': 'gld', 'CL=F': 'oil'}
prices = {}

for symbol, name in symbols_map.items():
    df = con.execute(f"""
        SELECT date, close FROM silver_macro_prices
        WHERE symbol = '{symbol}' ORDER BY date
    """).df()
    df['date'] = pd.to_datetime(df['date'])
    prices[name] = df.set_index('date')['close'].sort_index()

feat['ret_qqq_3m'] = prices['qqq'].pct_change(63)
feat['ret_qqq_6m'] = prices['qqq'].pct_change(126)
feat['qqq_spy_spread_6m'] = feat['ret_qqq_6m'] - feat['ret_spy_6m']

feat['ret_xle_3m'] = prices['xle'].pct_change(63)
feat['ret_xle_6m'] = prices['xle'].pct_change(126)

feat['ret_gld_3m'] = prices['gld'].pct_change(63)
feat['ret_gld_6m'] = prices['gld'].pct_change(126)

feat['ret_oil_1m'] = prices['oil'].pct_change(21)
feat['ret_oil_3m'] = prices['oil'].pct_change(63)
feat['ret_oil_6m'] = prices['oil'].pct_change(126)

# 1C. Breadth
breadth_df = con.execute("""
    SELECT date,
           AVG(CASE WHEN ret_252d > 0        THEN 1.0 ELSE 0.0 END) AS breadth_pos_12m,
           AVG(CASE WHEN ret_21d  > 0        THEN 1.0 ELSE 0.0 END) AS breadth_pos_1m,
           AVG(CASE WHEN pct_from_ath > -0.10 THEN 1.0 ELSE 0.0 END) AS breadth_near_ath,
           AVG(CASE WHEN rvol_252d > 0.30    THEN 1.0 ELSE 0.0 END) AS breadth_rvol_elevated
    FROM gold_panel
    WHERE index_type = 'S&P 500'
    GROUP BY date
    ORDER BY date
""").df()
breadth_df['date'] = pd.to_datetime(breadth_df['date'])
breadth = breadth_df.set_index('date')
feat = feat.join(breadth, how='left')

for col in breadth.columns:
    feat[col] = feat[col].ffill()

# 1D. Fear gauge
vix_df = con.execute("""
    SELECT date, close FROM silver_macro_prices
    WHERE symbol = 'VIX' ORDER BY date
""").df()
vix_df['date'] = pd.to_datetime(vix_df['date'])
vix = vix_df.set_index('date')['close'].sort_index()

feat['vix_level'] = vix
feat['vix_chg_21d'] = vix.pct_change(21)

hy_df = con.execute("""
    SELECT date, close FROM silver_macro_prices
    WHERE symbol = 'HY_OAS' ORDER BY date
""").df()
hy_df['date'] = pd.to_datetime(hy_df['date'])
hy = hy_df.set_index('date')['close'].sort_index()

feat['hy_spread'] = hy
feat['hy_spread_chg_21d'] = hy.pct_change(21)
feat['hy_spread_chg_63d'] = hy.pct_change(63)

y10_df = con.execute("""
    SELECT date, close FROM silver_macro_prices
    WHERE symbol = 'TNX_10Y' ORDER BY date
""").df()
y3m_df = con.execute("""
    SELECT date, close FROM silver_macro_prices
    WHERE symbol = 'IRX_3M' ORDER BY date
""").df()

y10_df['date'] = pd.to_datetime(y10_df['date'])
y3m_df['date'] = pd.to_datetime(y3m_df['date'])

y10 = y10_df.set_index('date')['close'].sort_index()
y3m = y3m_df.set_index('date')['close'].sort_index()

feat['ted_spread'] = y10 - y3m

print(f'Features engineered: {feat.shape[1]} columns, {feat.shape[0]} rows')
print(f'Date range: {feat.index.min()} to {feat.index.max()}')
print(f"\nFeature breakdown:")
print(f'  - SPY momentum & volatility: 6 features')
print(f'  - Asset class signals: 10 features (QQQ, XLE, GLD, oil)')
print(f'  - Market breadth: 4 features')
print(f'  - Fear gauge: 6 features (VIX, HY spread, TED spread)')
print(f'  Total: {feat.shape[1]} features')

## Build Labels

In [ ]:
# Labels: forward 63-day return
feat['fwd_ret_63d'] = spy.pct_change(63).shift(-63)
feat['trail_ret_126d'] = spy.pct_change(126)

def forward_regime(row):
    fwd = row['fwd_ret_63d']
    trail = row['trail_ret_126d']
    
    if pd.isna(fwd) or pd.isna(trail):
        return np.nan
    
    if fwd > 0.07:
        return 'EXPANSION'
    elif fwd < -0.05:
        return 'CONTRACTION'
    elif fwd > 0.0 and trail < -0.05:
        return 'RECOVERY'
    else:
        return 'LATE_CYCLE'

feat['regime'] = feat.apply(forward_regime, axis=1)

print(f'Regimes labeled. Distribution:')
print(feat['regime'].value_counts())

## Walk-Forward Cross-Validation

Train on past 5 years, test on next 1 year, roll forward

In [ ]:
# Clean data
df_clean = feat.dropna(subset=['regime']).copy()

# Walk-forward parameters
train_window = 252 * 5  # 5 years of training data
test_window = 252 * 1   # 1 year of testing data

# Find start date for first fold (need enough history)
start_idx = train_window

results = []

fold = 0
while start_idx + test_window < len(df_clean):
    fold += 1
    
    # Split data
    train_end_idx = start_idx
    test_end_idx = start_idx + test_window
    
    train_data = df_clean.iloc[:train_end_idx]
    test_data = df_clean.iloc[train_end_idx:test_end_idx]
    
    # Skip if not enough data
    if len(train_data) < train_window or len(test_data) < 100:
        start_idx += test_window
        continue
    
    # Prepare training data
    X_train = train_data.drop(['fwd_ret_63d', 'trail_ret_126d', 'regime'], axis=1)
    y_train = train_data['regime']
    
    # Drop NaN from training
    valid_idx = X_train.notna().all(axis=1)
    X_train = X_train[valid_idx]
    y_train = y_train[valid_idx]
    
    # Prepare test data
    X_test = test_data.drop(['fwd_ret_63d', 'trail_ret_126d', 'regime'], axis=1)
    y_test = test_data['regime']
    
    valid_idx_test = X_test.notna().all(axis=1)
    X_test = X_test[valid_idx_test]
    y_test = y_test[valid_idx_test]
    
    if len(X_test) == 0:
        start_idx += test_window
        continue
    
    # Train model
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_test_enc = le.transform(y_test)
    
    model = GradientBoostingClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, min_samples_leaf=20, validation_fraction=0.1,
        n_iter_no_change=30, random_state=42
    )
    
    model.fit(X_train, y_train_enc)
    
    # Evaluate
    train_acc = model.score(X_train, y_train_enc)
    test_acc = model.score(X_test, y_test_enc)
    
    baseline_acc = np.bincount(y_test_enc).max() / len(y_test_enc)  # Most common class
    
    results.append({
        'fold': fold,
        'train_period': f"{train_data.index[0].date()} to {train_data.index[-1].date()}",
        'test_period': f"{test_data.index[0].date()} to {test_data.index[-1].date()}",
        'train_acc': train_acc,
        'test_acc': test_acc,
        'baseline_acc': baseline_acc,
        'test_samples': len(X_test)
    })
    
    # Move forward
    start_idx += test_window

results_df = pd.DataFrame(results)

print(f'\n{"="*80}')
print(f'WALK-FORWARD VALIDATION RESULTS (26-feature baseline)')
print(f'{"="*80}')
print(results_df.to_string(index=False))

## Summary Statistics

In [ ]:
print(f'\n{"="*80}')
print(f'SUMMARY')
print(f'{"="*80}')
print(f'\nTotal folds: {len(results_df)}')
print(f'Date range: {results_df["test_period"].iloc[0].split(" to ")[0]} to {results_df["test_period"].iloc[-1].split(" to ")[1]}')

print(f'\nOut-of-Sample Accuracy (OOS):')
print(f'  Mean: {results_df["test_acc"].mean():.1%}')
print(f'  Std:  {results_df["test_acc"].std():.1%}')
print(f'  Min:  {results_df["test_acc"].min():.1%}')
print(f'  Max:  {results_df["test_acc"].max():.1%}')

print(f'\nComparison to Baseline (random guess):')
baseline_mean = results_df['baseline_acc'].mean()
print(f'  Baseline (most common class): {baseline_mean:.1%}')
print(f'  Model vs Baseline: +{(results_df["test_acc"].mean() - baseline_mean):.1%}')

print(f'\n⭐ KEY INSIGHT:')
print(f'  Walk-forward accuracy: {results_df["test_acc"].mean():.1%}')
print(f'  This is the REALISTIC expected performance.')
print(f'  Regime classification is genuinely hard: regime ≠ return.')
print(f'  Value is consistency & lead time, not perfect accuracy.')
print(f'  Use as ONE signal among many for portfolio tilting.')

## Visualization

In [ ]:
## Visualization